# Fifa Prediction model

Each training example will be a (22,9) matrix. Each row is a player, the first 11 rows are players from the home team while the last 11 rows are players from the away team. For each player, the following details are used:

1. age
2. is forward
3. is mid fielder
4. is defense
5. is goalkeeper
6. number tournaments
7. appearances
8. goals
9. average time per team

This matrix will be passed to the first layer of the neural network that should predict the 'quality' of each player in the team. There will be a ReLU activation for this network. This will be done by doing:

$$
p = xw^T + b \\
ap = max(0, p)
$$

The output of this will be a (22, 1) matrix. This will then be passed to the team wins prediction portion of the network. This has 1 layer with 2 units with ReLU activation as shown below:

$$
L = 1 \\
n^{[1]} = 2 \\
z = apw + b \\
y = max(0, z)
$$

The goal is to get a model that surpasses the following benchmarks:
- 75.42% accuracy on predicting a win
- 18% accuracy for predicting exact scores

In [2]:
# Imports
import numpy as np
import numpy.typing as npt

In [46]:
def f_x(
    players: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float
):
    """ 
    Function that will do a forward pass for a single training example in the example above.
    
    Args:
        players (ndarray) - a (22,9) numpy array with 9 features for 22 players from the home and away team
        w_p (ndarray) - a (1,9) numpy array with weights for player predictions
        b_p (scalar) - bias for player quality prediction
        w_t (ndarray) - a (2, 22) numpy array with the weights for the team score prediction
        b_t (scalar) - bias for the team score prediction
        
    Returns:
        scores (ndarray) - a (2,1) numpy array with the predicted home and away scores
        z_t (ndarray) - a (2,1) numpy array showing value before final activation
        a_p (ndarray) - cached a_t value,
        z_p (ndarray) - cached z_p value
    """
    z_p = np.matmul(players, w_p.T) + b_p
    a_p = np.maximum(0, z_p)
    z_t = np.matmul(w_t, a_p) + b_t
    a_t = np.maximum(0, z_t)
    return (a_t, z_t, a_p, z_p)
    

In [47]:
# Forward pass is working
players_1 = np.ones((1, 9))
players_1 = np.tile(players_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((1, 22))
w_t = np.tile(w_t, (2, 1))
b_t = 0
result_1 = f_x(
    players_1,
    w_p,
    b_p,
    w_t,
    b_t
)
print(result_1)

(array([[198.],
       [198.]]), array([[198.],
       [198.]]), array([[9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.]]), array([[9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.],
       [9.]]))


In [48]:
def L(y_pred, y):
    """
    Returns the means squared error of a single example
    
    Args:
        y_pred (ndarray) - predicted home away goal scores
        y (ndarray) - actual home away goal scores
        
    Returns:
        error (scalar) - mean squared error loss
    """
    diff = y_pred - y
    return np.sum(diff ** 2)

In [49]:
print(L(np.array([1,1]), np.array([0,0])))

2


In [67]:
def calculate_gradients(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Function to calculate the gradients used in gradient descent
    
    Args:
        X (ndarray) - a (m,22,9) array with m training examples
        Y (ndarray) - a (m, 2, 1) array with labels for m training examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
        
    Returns:
        gradients (list) - list of gradients of parameters
    """
    dw_t = np.zeros(w_t.shape)
    db_t = 0
    dw_p = np.zeros(w_p.shape)
    db_p = 0
    J = 0
    
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, z_t_i, a_p_i, z_p_i) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        
        # Get derivative
        da_t_i = 2 * (y_pred_i - y_i)
        dz_t_i = np.where(z_t_i < 0, 0, da_t_i)
        print(dz_t_i)
        dw_t += a_p_i.T * dz_t_i
        db_t += np.sum(dz_t_i)
        da_p_i = np.matmul(w_t.T, dz_t_i)
        dz_p_i = np.where(z_p_i < 0, 0, da_p_i)
        dw_p += np.matmul(dz_p_i.T, x_i)
        db_p += np.sum(dz_p_i)
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
        
    dw_t /= m
    db_t /= m
    dw_p /= m
    db_p /= m
    J /= m
    return (dw_p, db_p, dw_t, db_t, J)

In [68]:
x_train_1 = np.ones((1, 9))
x_train_1 = np.tile(x_train_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((1, 22))
w_t = np.tile(w_t, (2, 1))
b_t = 0
X_train = np.array([
    x_train_1
])
Y_train = np.array([
    np.array([
        [2],
        [2]
    ])
])

print(calculate_gradients(
    X_train,
    w_p,
    b_p,
    w_t,
    b_t,
    Y_train
))

[[392.]
 [392.]]
(array([[17248., 17248., 17248., 17248., 17248., 17248., 17248., 17248.,
        17248.]]), np.float64(17248.0), array([[3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528.,
        3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528.,
        3528., 3528., 3528., 3528.],
       [3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528.,
        3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528., 3528.,
        3528., 3528., 3528., 3528.]]), np.float64(784.0), np.float64(76832.0))
